# 07 — Pipeline live de actualización del Mundial 2026

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 7 — Predicciones dinámicas: re-simular condicionado a resultados reales

## Objetivo

1. **Optimizar el simulador** con `PrecomputedPredictor`: reduce 10K sims de **45 min → ~2-3 min** al precomputar los 2.256 pares posibles (48×47).
2. **API simple** para fijar resultados reales del Mundial:
   ```python
   fr = FixedResults()
   fr.add_group_match('Group_J', 'Argentina', 'Algeria', home_score=3, away_score=0)
   fr.add_knockout_result(match_id=89, winner='Argentina')
   ```
3. **Re-simulación condicionada**: las probabilidades se actualizan dado el estado real del torneo.
4. **Comparación deltas**: ¿cómo cambia P(Argentina campeón) después de cada partido?

## Cómo se usa en la práctica durante el Mundial

Cada vez que termine un partido (a partir del 11 de junio):

1. Agregás el resultado al `FixedResults`.
2. Corrés `monte_carlo(predictor, fixed_results=fr)` (~3 min).
3. Mirás las nuevas probabilidades.

Para Fase 8 podemos automatizar este flujo (scheduled task que lee resultados de una API y re-simula).

---
## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import sys, json, pickle, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_INTERIM = paths.data_interim
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

from simulation.bracket import WC2026_GROUPS, assert_bracket_consistency
from simulation.simulator import (
    monte_carlo, PrecomputedPredictor, FixedResults,
)
from data.feature_engineering import build_feature_matrix, get_feature_columns
from models.baselines import LogisticRegressionBaseline

assert_bracket_consistency()
print('✓ Bracket OK')

---
## 2. Cargar datos + entrenar LogReg (mismo flujo que Fase 6)

In [ ]:
import math

df_int = pd.read_parquet(DATA_INTERIM / 'international_matches_with_elo.parquet')
df_int['date'] = pd.to_datetime(df_int['date'])
df_sorted = df_int.sort_values('date').reset_index(drop=True).copy()

# Rolling features (idéntico a Fase 4a/6)
team_history = defaultdict(list)
WINDOW = 5
def rmean(history, idx):
    recent = history[-WINDOW:]
    if not recent: return math.nan
    return sum(x[idx] for x in recent) / len(recent)

home_form, away_form, home_goals, away_goals = [], [], [], []
for _, row in df_sorted.iterrows():
    h, a = row.home_team, row.away_team
    hs, as_ = row.home_score, row.away_score
    home_form.append(rmean(team_history[h], 0))
    away_form.append(rmean(team_history[a], 0))
    home_goals.append(rmean(team_history[h], 1))
    away_goals.append(rmean(team_history[a], 1))
    if pd.notna(hs) and pd.notna(as_):
        if hs > as_:   h_pts, a_pts = 3, 0
        elif hs < as_: h_pts, a_pts = 0, 3
        else:          h_pts, a_pts = 1, 1
        team_history[h].append((h_pts, float(hs)))
        team_history[a].append((a_pts, float(as_)))
df_sorted['home_form_pts_5']     = home_form
df_sorted['away_form_pts_5']     = away_form
df_sorted['home_recent_goals_5'] = home_goals
df_sorted['away_recent_goals_5'] = away_goals

# Last state per team
WC_TEAMS = sorted({t for grp in WC2026_GROUPS.values() for t in grp})
def latest_state(team):
    rows = df_sorted[(df_sorted.home_team == team) | (df_sorted.away_team == team)]
    if len(rows) == 0: return None
    r = rows.iloc[-1]
    if r.home_team == team:
        return {'team': team, 'elo': float(r.home_elo_after) if pd.notna(r.home_elo_after) else float(r.home_elo_before),
                'form_pts': r.home_form_pts_5, 'recent_goals': r.home_recent_goals_5}
    return {'team': team, 'elo': float(r.away_elo_after) if pd.notna(r.away_elo_after) else float(r.away_elo_before),
            'form_pts': r.away_form_pts_5, 'recent_goals': r.away_recent_goals_5}

team_features = {}
for t in WC_TEAMS:
    st = latest_state(t)
    if st is None:
        team_features[t] = {'team': t, 'elo': 1500.0, 'form_pts': 1.0, 'recent_goals': 1.0}
    else:
        team_features[t] = st
print(f'✓ Features cargadas para {len(team_features)} equipos')

In [ ]:
# Entrenar LogReg
X_full, y_full, df_full = build_feature_matrix(
    df_sorted, windows_form=(5, 10), h2h_window=5, min_date='2014-01-01',
)
logreg = LogisticRegressionBaseline()
logreg.fit(X_full, y_full)
FEATURE_COLS = get_feature_columns(df_full)
print(f'✓ LogReg entrenado sobre {len(X_full):,} partidos')

In [ ]:
def build_feature_vector(team_a, team_b, venue='neutral'):
    fa = team_features.get(team_a, {'elo': 1500, 'form_pts': 1.0, 'recent_goals': 1.0})
    fb = team_features.get(team_b, {'elo': 1500, 'form_pts': 1.0, 'recent_goals': 1.0})
    elo_diff = fa['elo'] - fb['elo']
    raw = {
        'tournament_class': 'world_cup_final',
        'neutral': 1 if venue == 'neutral' else 0,
        'home_elo': fa['elo'], 'away_elo': fb['elo'], 'elo_diff': elo_diff,
        'expected_home_win_prob': 1 / (1 + 10 ** (-elo_diff / 400)),
        'home_rest_days': 7, 'away_rest_days': 7,
        'home_form5_pts': fa['form_pts'] if pd.notna(fa['form_pts']) else 1.0,
        'home_form5_gf':  fa['recent_goals'] if pd.notna(fa['recent_goals']) else 1.0,
        'home_form5_ga': 1.0, 'home_form5_gd': 0.0, 'home_form5_n': 5,
        'away_form5_pts': fb['form_pts'] if pd.notna(fb['form_pts']) else 1.0,
        'away_form5_gf':  fb['recent_goals'] if pd.notna(fb['recent_goals']) else 1.0,
        'away_form5_ga': 1.0, 'away_form5_gd': 0.0, 'away_form5_n': 5,
        'home_form10_pts': fa['form_pts'] if pd.notna(fa['form_pts']) else 1.0,
        'home_form10_gf':  fa['recent_goals'] if pd.notna(fa['recent_goals']) else 1.0,
        'home_form10_ga': 1.0, 'home_form10_gd': 0.0, 'home_form10_n': 10,
        'away_form10_pts': fb['form_pts'] if pd.notna(fb['form_pts']) else 1.0,
        'away_form10_gf':  fb['recent_goals'] if pd.notna(fb['recent_goals']) else 1.0,
        'away_form10_ga': 1.0, 'away_form10_gd': 0.0, 'away_form10_n': 10,
        'h2h_n_matches': 0, 'h2h_home_wins': 0, 'h2h_draws': 0,
        'h2h_away_wins': 0, 'h2h_avg_gd_for_home': 0.0,
    }
    df_row = pd.DataFrame([raw])
    return df_row

def logreg_predictor_raw(team_a, team_b, venue='neutral'):
    x = build_feature_vector(team_a, team_b, venue=venue)
    return logreg.predict_proba(x)[0]

# Sanity
print('Argentina vs France (sanity):', logreg_predictor_raw('Argentina', 'France'))

---
## 3. PrecomputedPredictor — 100× speedup

Pre-computamos las 2.256 predicciones (48 × 47) una sola vez. Después cada simulación hace solo dict lookups.

In [ ]:
t0 = time.time()
predictor = PrecomputedPredictor(logreg_predictor_raw, WC_TEAMS, venue='neutral', verbose=True)
elapsed = time.time() - t0
print(f'✓ Predictor precomputado en {elapsed:.1f}s ({len(predictor.table)} pares)')

---
## 4. Baseline: predicción inicial (sin resultados fijos)

In [ ]:
t0 = time.time()
agg_baseline = monte_carlo(predictor, n_iters=10_000, seed=42, progress=True)
elapsed = time.time() - t0
print(f'\n✓ Baseline ({agg_baseline.n_iters:,} sims) en {elapsed:.1f}s ({agg_baseline.n_iters/elapsed:.1f} sim/s)')
df_baseline = agg_baseline.to_dataframe()
print('\nTop-10 baseline:')
print(df_baseline.head(10).to_string(index=False, float_format=lambda x: f'{x:.4f}'))

---
## 5. API de actualización en vivo

Esta es la función que vas a usar durante el Mundial real. Cada vez que termine un partido, llamala y guardá las nuevas probabilidades.

In [ ]:
def update_and_simulate(fixed_results: FixedResults, n_iters: int = 10000, seed: int = 42):
    """Re-simulate the WC given the real-world results known so far."""
    print(f'Fixed: {fixed_results.summary()}')
    t0 = time.time()
    agg = monte_carlo(predictor, n_iters=n_iters, seed=seed, progress=False,
                       fixed_results=fixed_results)
    elapsed = time.time() - t0
    print(f'✓ {n_iters:,} sims condicionadas en {elapsed:.1f}s')
    return agg

def compare_predictions(df_before, df_after, top_n=15):
    """Show delta in P_champion between two predictions."""
    merged = df_before.merge(df_after, on='Equipo' if 'Equipo' in df_before.columns else 'team',
                              suffixes=('_before', '_after'))
    merged['Δ P_champion'] = merged['P_champion_after'] - merged['P_champion_before']
    merged = merged.sort_values('P_champion_after', ascending=False).head(top_n)
    cols = [c for c in merged.columns if c in ['Equipo', 'team', 'P_champion_before', 'P_champion_after', 'Δ P_champion']]
    return merged[cols]

# Ensure 'Equipo' col exists
df_baseline = agg_baseline.to_dataframe()
if 'team' in df_baseline.columns and 'Equipo' not in df_baseline.columns:
    df_baseline = df_baseline.rename(columns={'team': 'Equipo'})
df_baseline.head(3)

---
## 6. Demo: ¿qué pasa si Argentina arrasa su grupo?

Simulamos los 3 partidos hipotéticos del grupo J de Argentina (Algeria, Austria, Jordan) con victorias contundentes.  
Esperamos que P(Argentina campeón) suba.

In [ ]:
fr_demo = FixedResults()
fr_demo.add_group_match('Group_J', home='Argentina', away='Algeria', home_score=3, away_score=0)
fr_demo.add_group_match('Group_J', home='Argentina', away='Austria', home_score=2, away_score=0)
fr_demo.add_group_match('Group_J', home='Argentina', away='Jordan',  home_score=4, away_score=0)

agg_demo = update_and_simulate(fr_demo, n_iters=10_000)
df_demo = agg_demo.to_dataframe().rename(columns={'team': 'Equipo'})

diff = compare_predictions(df_baseline, df_demo, top_n=20)
print('\nTop-20 ranked por P_champion DESPUÉS de fijar grupo J pro-Argentina:')
print(diff.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# Visualizar el delta
top12 = diff.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(top12))
ax.barh(y - 0.2, top12['P_champion_before'] * 100, height=0.4, label='Antes (baseline)', color='#aaaaaa')
ax.barh(y + 0.2, top12['P_champion_after'] * 100, height=0.4, label='Después (grupo J Arg arrasa)', color='#1f77b4')
ax.set_yticks(y); ax.set_yticklabels(top12['Equipo'])
ax.set_xlabel('P(campeón) (%)')
ax.set_title('Demo: cambio en probabilidades tras fijar el grupo J')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / 'wc2026_live_update_demo.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Demo extrema: ¿y si Argentina pierde su primer partido?

Esta es la prueba ácida del simulador condicional.

In [ ]:
fr_extreme = FixedResults()
fr_extreme.add_group_match('Group_J', home='Argentina', away='Algeria', home_score=0, away_score=2)

agg_extreme = update_and_simulate(fr_extreme, n_iters=10_000)
df_extreme = agg_extreme.to_dataframe().rename(columns={'team': 'Equipo'})

merged = df_baseline.merge(df_extreme, on='Equipo', suffixes=('_before', '_after'))
merged['Δ P_champion'] = merged['P_champion_after'] - merged['P_champion_before']
merged['Δ P_group'] = merged['P_group_advance_after'] - merged['P_group_advance_before']

# Mostrar a Argentina específicamente
arg_row = merged[merged['Equipo'] == 'Argentina'].iloc[0]
print(f'=== Argentina pierde su primer partido (0-2 vs Algeria) ===')
print(f'P(pasa grupo) ANTES: {arg_row["P_group_advance_before"]:.4f}')
print(f'P(pasa grupo) DESPUÉS: {arg_row["P_group_advance_after"]:.4f}  (Δ = {arg_row["Δ P_group"]:+.4f})')
print(f'P(campeón) ANTES: {arg_row["P_champion_before"]:.4f}')
print(f'P(campeón) DESPUÉS: {arg_row["P_champion_after"]:.4f}  (Δ = {arg_row["Δ P_champion"]:+.4f})')

# Argelia
alg_row = merged[merged['Equipo'] == 'Algeria'].iloc[0]
print(f'\n=== Algeria gana su primer partido (2-0 vs Argentina) ===')
print(f'P(pasa grupo) Algeria: {alg_row["P_group_advance_before"]:.4f} → {alg_row["P_group_advance_after"]:.4f}  (Δ = {alg_row["Δ P_group"]:+.4f})')

---
## 8. Persistir un `FixedResults` vacío para inicializar el Mundial

Cuando empiece el Mundial real (11 de junio), vas a:
1. Cargar este archivo.
2. Cada vez que termine un partido: `fr.add_group_match(...)` o `fr.add_knockout_result(...)`.
3. Re-correr `update_and_simulate(fr)`.
4. Guardar el `FixedResults` actualizado para el próximo partido.

In [ ]:
# Empty FixedResults — punto de partida del torneo
fr_initial = FixedResults()
live_state_path = ROOT / 'data' / 'live' / 'wc2026_state.pkl'
live_state_path.parent.mkdir(exist_ok=True)
with open(live_state_path, 'wb') as f:
    pickle.dump(fr_initial, f)
print(f'✓ Estado inicial vacío guardado: {live_state_path}')

# Helper functions para uso interactivo
def load_state():
    with open(live_state_path, 'rb') as f:
        return pickle.load(f)

def save_state(fr):
    with open(live_state_path, 'wb') as f:
        pickle.dump(fr, f)
    print(f'✓ Estado guardado: {fr.summary()}')

print('\nFunciones disponibles:')
print('  fr = load_state()')
print('  fr.add_group_match(...)')
print('  save_state(fr)')
print('  agg = update_and_simulate(fr)')

---
## 9. Tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', str(ROOT / 'tests' / 'test_simulation.py'), '-v', '--tb=short'],
    capture_output=True, text=True,
)
print(result.stdout[-3500:])
print(f'Exit code: {result.returncode}')

---
## 10. Conclusiones de Fase 7

Llenar:

- [ ] Tiempo de precomputado del predictor: _____ s
- [ ] Tiempo de 10.000 sims con `PrecomputedPredictor`: _____ s (esperamos <300s)
- [ ] Speedup vs Fase 6: _____ × (Fase 6 fueron 2753s)
- [ ] P(Argentina campeón) en baseline: _____
- [ ] P(Argentina campeón) si arrasa grupo J: _____ (Δ = _____)
- [ ] P(Argentina campeón) si pierde primero (0-2 Algeria): _____ (Δ = _____)
- [ ] Tests pasados / total: _____ / 22

**Estado del proyecto:** todo el pipeline funciona end-to-end:
- Fase 0-2: datos + vocab.
- Fase 3-4: Transformer pre-entrenado + fine-tuneado.
- Fase 5: análisis de embeddings (hallazgo principal: ordinalidad ELO).
- Fase 6: predicciones iniciales WC 2026.
- Fase 7: pipeline live para actualizar predicciones día a día.

**Falta solo Fase 8:** verificación final (tests integrales, reproducibilidad, paper).